### 경로 + 팀 평가 함수 로드

In [1]:
import sys, json
from pathlib import Path

REPO = Path(r"C:\Users\chltp\Documents\NLP\Study-service_basic")   # 팀 저장소
sys.path.append(str(REPO / "team" / "eval"))                       # evaluate.py가 있는 폴더를 import 경로에 추가

from evaluate import evaluate, load_questions, build_retriever, page_of

questions = load_questions(REPO / "team" / "eval" / "questions.jsonl")
print(len(questions), questions[0]["question"])


25 비밀 키를 소스 코드에 직접 적어 두면 어떤 위험이 있어?


###  후보 20개 검색기

In [2]:
vector_20 = build_retriever(REPO / "team" / "wiki", chunk_size=500, chunk_overlap=50, k=20)

base = evaluate(vector_20, questions, k=5, show_misses=False)

c:\Users\chltp\Documents\NLP\tool스터디\week4\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 16071.39it/s]


페이지 31개 -> 청크 174개 (chunk_size=500, overlap=50)

질문 25개 | Recall@5 = 1.000 | MRR@5 = 0.759 | 평균 검색 시간 252.0 ms


### 리랭커 로드 + 질문 하나로 점수 보기

In [3]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512)

q = next(x for x in questions if x["id"] == "q25")["question"]
cands = vector_20.invoke(q)                                        # 1차: 벡터로 후보 20개
scores = reranker.predict([(q, d.page_content) for d in cands])    # 2차: (질문, 청크) 쌍마다 점수

print("벡터 순위 상위 5:", [page_of(d) for d in cands[:5]])
print("\n리랭커 순위 상위 5:")
for d, s in sorted(zip(cands, scores), key=lambda x: -x[1])[:5]:
    print(f"  {s:.3f}`  {page_of(d)}")


Loading weights: 100%|██████████| 393/393 [00:01<00:00, 274.27it/s]


벡터 순위 상위 5: ['container-commands_mingi', 'image-vs-container_huiji', 'image-vs-container_doyu', 'image-container-layer-registry_mingi', '1-docker-concepts_sejin']

리랭커 순위 상위 5:
  0.968`  image-vs-container_doyu
  0.967`  container-commands_mingi
  0.967`  image-vs-container_huiji
  0.816`  image-container-layer-registry_mingi
  0.732`  container-commands_mingi


### 리랭킹 검색기 + 전체 채점

In [4]:
class RerankRetriever:
    """1차 벡터 후보 → 리랭커 점수로 다시 정렬 → 상위 top_n개"""
    def __init__(self, base, reranker, top_n=5):
        self.base, self.reranker, self.top_n = base, reranker, top_n

    def invoke(self, question):
        cands = self.base.invoke(question)                                          # 후보 20개
        scores = self.reranker.predict([(question, d.page_content) for d in cands])
        ranked = sorted(zip(cands, scores), key=lambda x: -x[1])
        return [d for d, _ in ranked[: self.top_n]]

rerank_20 = RerankRetriever(vector_20, reranker)
rr = evaluate(rerank_20, questions, k=5, show_misses=True)



질문 25개 | Recall@5 = 1.000 | MRR@5 = 0.808 | 평균 검색 시간 26972.7 ms

정답을 1위로 못 찾은 질문
- q01 정답 순위 3 | 비밀 키를 소스 코드에 직접 적어 두면 어떤 위험이 있어?
    정답: ['api-key-env_mingi']  검색: ['rag-evaluation_mingi', '1-docker-concepts_sejin', 'api-key-env_mingi', 'docker-compose_mingi', 'redis-ttl-cache_mingi']
- q07 정답 순위 2 | 여러 컨테이너의 실행 설정을 파일 하나에 적어 한 번에 띄우는 방법은?
    정답: ['docker-compose_mingi']  검색: ['rag-evaluation_mingi', 'docker-compose_mingi', 'container-commands_mingi', '1-docker-concepts_sejin', '1-docker-concepts_sejin']
- q09 정답 순위 2 | 이미지를 만들 때 실행되는 명령과 컨테이너가 시작될 때 실행되는 명령은 어떻게 달라?
    정답: ['dockerfile-instructions_mingi', '1-docker-concepts_sejin']  검색: ['image-vs-container_doyu', 'dockerfile-instructions_mingi', 'image-container-layer-registry_mingi', '1-docker-concepts_sejin', 'container-commands_mingi']
- q15 정답 순위 5 | 컨테이너가 꺼진 뒤에도 안에서 만든 결과 파일을 남기려면?
    정답: ['volume-port_mingi', '1-docker-concepts_sejin']  검색: ['container-commands_mingi', 'embedding-sentence-transformers_mingi', 'rag-evaluat

### 저장

In [5]:
import json
rr["config"] = {"base": "bge-m3 500/50", "candidates": 20, "reranker": "BAAI/bge-reranker-v2-m3", "top_n": 5}
with open("results/rerank_20.json", "w", encoding="utf-8") as f:
    json.dump(rr, f, ensure_ascii=False, indent=2)


### 후보 10개로 재측정

In [4]:
class RerankRetriever:
    """1차 벡터 후보 n_cands개 → 리랭커 점수로 다시 정렬 → 상위 top_n개"""
    def __init__(self, base, reranker, n_cands=20, top_n=5):
        self.base, self.reranker = base, reranker
        self.n_cands, self.top_n = n_cands, top_n

    def invoke(self, question):
        cands = self.base.invoke(question)[: self.n_cands]                          # 앞 n_cands개만 후보로
        scores = self.reranker.predict([(question, d.page_content) for d in cands])
        ranked = sorted(zip(cands, scores), key=lambda x: -x[1])
        return [d for d, _ in ranked[: self.top_n]]


In [5]:
rerank_10 = RerankRetriever(vector_20, reranker, n_cands=10)
rr10 = evaluate(rerank_10, questions, k=5, show_misses=True)

rr10["config"] = {"base": "bge-m3 500/50", "candidates": 10, "reranker": "BAAI/bge-reranker-v2-m3", "top_n": 5}
with open("results/rerank_10.json", "w", encoding="utf-8") as f:
    json.dump(rr10, f, ensure_ascii=False, indent=2)



질문 25개 | Recall@5 = 1.000 | MRR@5 = 0.830 | 평균 검색 시간 10259.9 ms

정답을 1위로 못 찾은 질문
- q01 정답 순위 3 | 비밀 키를 소스 코드에 직접 적어 두면 어떤 위험이 있어?
    정답: ['api-key-env_mingi']  검색: ['rag-evaluation_mingi', '1-docker-concepts_sejin', 'api-key-env_mingi', 'docker-compose_mingi', 'redis-ttl-cache_mingi']
- q07 정답 순위 2 | 여러 컨테이너의 실행 설정을 파일 하나에 적어 한 번에 띄우는 방법은?
    정답: ['docker-compose_mingi']  검색: ['rag-evaluation_mingi', 'docker-compose_mingi', 'container-commands_mingi', '1-docker-concepts_sejin', '1-docker-concepts_sejin']
- q09 정답 순위 2 | 이미지를 만들 때 실행되는 명령과 컨테이너가 시작될 때 실행되는 명령은 어떻게 달라?
    정답: ['dockerfile-instructions_mingi', '1-docker-concepts_sejin']  검색: ['image-vs-container_doyu', 'dockerfile-instructions_mingi', 'image-container-layer-registry_mingi', '1-docker-concepts_sejin', 'container-commands_mingi']
- q15 정답 순위 4 | 컨테이너가 꺼진 뒤에도 안에서 만든 결과 파일을 남기려면?
    정답: ['volume-port_mingi', '1-docker-concepts_sejin']  검색: ['container-commands_mingi', 'embedding-sentence-transformers_mingi', 'rag-evaluat